In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [16]:
df = pd.read_csv("../data/processed/ratnapark/ratnapark_pm1_pm25_pm10_daily.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)

display(df.head())

Shape: (1887, 4)


,date,pm1,pm2_5,pm10
0,2016-08-25,144.026083,187.599837,269.346300
1,2016-08-27,121.881565,159.537520,230.975440
2,2016-08-28,98.196718,124.696523,171.362190
3,2016-08-29,54.083770,63.915300,78.042140
4,2016-09-07,127.947061,168.501950,245.585955


In [17]:
print(df.columns.tolist())

['date', 'pm1', 'pm2_5', 'pm10']


In [18]:
print(df.isna().sum())

date     0
pm1      0
pm2_5    0
pm10     0
dtype: int64


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1887 entries, 0 to 1886
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    1887 non-null   datetime64[us]
 1   pm1     1887 non-null   float64       
 2   pm2_5   1887 non-null   float64       
 3   pm10    1887 non-null   float64       
dtypes: datetime64[us](1), float64(3)
memory usage: 59.1 KB


In [20]:
full_dates = pd.date_range(
    start=df["date"].min(),
    end=df["date"].max(),
    freq="D"
)

df_daily = (
    df.set_index("date")
      .reindex(full_dates)
      .rename_axis("date")
      .reset_index()
)

In [21]:
print("Original rows:", len(df))
print("Calendar rows:", len(df_daily))

Original rows: 1887
Calendar rows: 3416


In [22]:
display(df_daily[(df_daily["date"] >= "2016-09-25") &(df_daily["date"] <= "2016-09-29")])

,date,pm1,pm2_5,pm10
31,2016-09-25,84.983893,107.614070,146.479725
32,2016-09-26,53.756037,66.504697,89.382590
33,2016-09-27,NaN,NaN,NaN
34,2016-09-28,75.782110,92.105213,114.999075
35,2016-09-29,90.478135,110.280357,141.709305


In [23]:
df_daily["target_pm2_5"] = df_daily["pm2_5"].shift(-1)

In [24]:
pollutants = ["pm1", "pm2_5", "pm10"]

for pollutant in pollutants:
    for lag in range(1, 16):
        df_daily[f"{pollutant}_lag_{lag}"] = (
            df_daily[pollutant].shift(lag)
        )

In [25]:
for pollutant in pollutants:
    for window in [3, 7, 15]:
        df_daily[f"{pollutant}_rolling_mean_{window}"] = (
            df_daily[pollutant]
            .shift(1)
            .rolling(window)
            .mean()
        )

In [28]:
print("Total columns:", len(df_daily.columns))

display(df_daily.head())

Total columns: 59


,date,pm1,pm2_5,pm10,target_pm2_5,pm1_lag_1,pm1_lag_2,pm1_lag_3,pm1_lag_4,pm1_lag_5,...,pm10_lag_15,pm1_rolling_mean_3,pm1_rolling_mean_7,pm1_rolling_mean_15,pm2_5_rolling_mean_3,pm2_5_rolling_mean_7,pm2_5_rolling_mean_15,pm10_rolling_mean_3,pm10_rolling_mean_7,pm10_rolling_mean_15
0,2016-08-25,144.026083,187.599837,269.34630,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-08-26,NaN,NaN,NaN,159.537520,144.026083,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-08-27,121.881565,159.537520,230.97544,124.696523,NaN,144.026083,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-08-28,98.196718,124.696523,171.36219,63.915300,121.881565,NaN,144.026083,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-08-29,54.083770,63.915300,78.04214,NaN,98.196718,121.881565,NaN,144.026083,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
print(df_daily.columns.tolist())

['date', 'pm1', 'pm2_5', 'pm10', 'target_pm2_5', 'pm1_lag_1', 'pm1_lag_2', 'pm1_lag_3', 'pm1_lag_4', 'pm1_lag_5', 'pm1_lag_6', 'pm1_lag_7', 'pm1_lag_8', 'pm1_lag_9', 'pm1_lag_10', 'pm1_lag_11', 'pm1_lag_12', 'pm1_lag_13', 'pm1_lag_14', 'pm1_lag_15', 'pm2_5_lag_1', 'pm2_5_lag_2', 'pm2_5_lag_3', 'pm2_5_lag_4', 'pm2_5_lag_5', 'pm2_5_lag_6', 'pm2_5_lag_7', 'pm2_5_lag_8', 'pm2_5_lag_9', 'pm2_5_lag_10', 'pm2_5_lag_11', 'pm2_5_lag_12', 'pm2_5_lag_13', 'pm2_5_lag_14', 'pm2_5_lag_15', 'pm10_lag_1', 'pm10_lag_2', 'pm10_lag_3', 'pm10_lag_4', 'pm10_lag_5', 'pm10_lag_6', 'pm10_lag_7', 'pm10_lag_8', 'pm10_lag_9', 'pm10_lag_10', 'pm10_lag_11', 'pm10_lag_12', 'pm10_lag_13', 'pm10_lag_14', 'pm10_lag_15', 'pm1_rolling_mean_3', 'pm1_rolling_mean_7', 'pm1_rolling_mean_15', 'pm2_5_rolling_mean_3', 'pm2_5_rolling_mean_7', 'pm2_5_rolling_mean_15', 'pm10_rolling_mean_3', 'pm10_rolling_mean_7', 'pm10_rolling_mean_15']


In [30]:
df_model = df_daily.dropna().copy()

print("Calendar rows:", len(df_daily))
print("Usable modeling rows:", len(df_model))

Calendar rows: 3416
Usable modeling rows: 1409


In [31]:
print(df_model.isna().sum().sum())

0


In [32]:
display(df_model[[
            "date",
            "pm2_5",
            "pm2_5_lag_1",
            "pm2_5_lag_15",
            "target_pm2_5"
        ]
    ].head(10)
)

,date,pm2_5,pm2_5_lag_1,pm2_5_lag_15,target_pm2_5
49,2016-10-13,107.120657,70.023423,92.105213,110.907640
50,2016-10-14,110.907640,107.120657,110.280357,120.285923
51,2016-10-15,120.285923,110.907640,63.211730,129.241130
52,2016-10-16,129.241130,120.285923,66.716017,124.030080
53,2016-10-17,124.030080,129.241130,118.627510,116.180793
54,2016-10-18,116.180793,124.030080,123.281350,148.270110
55,2016-10-19,148.270110,116.180793,132.119323,143.660533
56,2016-10-20,143.660533,148.270110,115.576427,132.996460
57,2016-10-21,132.996460,143.660533,95.507687,159.309977
58,2016-10-22,159.309977,132.996460,106.309650,152.326270
